# ASG Airlines — Local Data Pipeline

This notebook runs the reproducible local pipeline from source profiling through cleaned, PII-safe curated data and Gold analytical tables. The source workbook is expected at `data/raw/UseCase - Airlines.xlsx`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

print(f"Project root: {ROOT}")

## 1. Profile the source

The profile records row counts, missing values, duplicates, malformed flight IDs, overnight candidates, duration issues, foreign-key checks, and PII columns.

In [ ]:
from profile import profile_workbook

profile = profile_workbook(ROOT / "data" / "raw" / "UseCase - Airlines.xlsx")
profile

## 2. Transform to curated/Silver

Text is normalized, timestamps are parsed, overnight arrivals are adjusted, invalid records are quarantined, and booking/passenger PII is removed or hashed.

In [ ]:
from transform import main as transform_pipeline

transform_pipeline()

## 3. Build Gold facts, dimensions, and KPIs

In [ ]:
from gold import build_gold

tables = build_gold()

In [ ]:
import pandas as pd

row_counts = pd.DataFrame({"table": list(tables), "rows": [len(frame) for frame in tables.values()]})
row_counts

## 4. Reconcile executive KPIs

In [ ]:
tables["agg_executive_kpis"]

## 5. PII serving-layer check

Only safe analytical fields may reach curated and Gold serving outputs.

In [ ]:
blocked = {"passenger_id", "passport_number", "emergency_contact_name", "emergency_contact_phone", "email", "phone", "aadhaar_id", "first_name", "last_name"}
checks = {}
for name in ["bookings", "passengers_safe"]:
    columns = set(pd.read_csv(ROOT / "data" / "curated" / f"{name}.csv", nrows=0).columns)
    checks[f"curated/{name}"] = sorted(columns & blocked)
columns = set(pd.read_csv(ROOT / "data" / "gold" / "fact_bookings.csv", nrows=0).columns)
checks["gold/fact_bookings"] = sorted(columns & blocked)
checks